# 🤖 Module 2: Autonomous Agentic Patterns & Tool Execution

This module builds an **autonomous customer service and returns agent** for an e-commerce platform. When a customer submits an inquiry or return request, the agent queries the order database, evaluates product-specific return policies, calculates net refunds, and enforces financial safety limits (auto-approving small refunds under $500 while routing high-value amounts for human manager sign-off).

### 🎯 What This Demonstrates (The 4 Core Patterns):
1. **Tools & Dynamic Execution**: Gives the AI "hands" to query order records, verify policies, and compute math.
2. **Human-in-the-Loop (HITL) Gate**: Auto-approves refunds $\le$ $500 while safely pausing high-value payouts (> $500) for manager sign-off.
3. **ReAct Reasoning Loop**: Industry-standard reasoning cycle: `Thought` ➔ `Action` ➔ `Observation` ➔ `Final Answer`.
4. **Plan-and-Solve + Self-Reflection**: Decomposes multi-step inquiries and verifies conditions before finalizing responses.

In [1]:
import sys
import os
import importlib

# Ensure project root is in sys.path
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

tools_mod = importlib.import_module("02_agentic_patterns.tools")
react_mod = importlib.import_module("02_agentic_patterns.react_agent")
plan_mod = importlib.import_module("02_agentic_patterns.plan_and_solve")

print("✅ All Module 2 Agentic functions imported successfully!")

✅ All Module 2 Agentic functions imported successfully!


## 1. Test Available Agent Tools

Tools give the AI "hands" to query your database and perform accurate calculations instead of guessing.

In [ ]:
# 1. Look up an existing customer order
order = tools_mod.lookup_order("ORD-2026-1001")
print(f"📦 Order Found: {order['item_name']} | Price: ${order['unit_price_usd']} | Customer: {order['customer_name']}")

# 2. Check policy restrictions for Laptops
policy = tools_mod.check_return_policy("Laptops")
print(f"📜 Policy Window: {policy['return_window_days']} days | Max Runtime: {policy['max_runtime_hours']} hrs | Restocking Fee: {policy['restocking_fee_pct']*100:.0f}%")

# 3. Calculate refund breakdown
refund_math = tools_mod.calculate_refund(order['unit_price_usd'], restocking_fee_pct=policy['restocking_fee_pct'])
print(f"💵 Net Refund: ${refund_math['net_refund_usd']} (Deducted fee: ${refund_math['restocking_fee_usd']})")

## 2. Human-in-the-Loop (HITL) Safety Guard

**The Rule:**
* Small refunds ($≤$ $500) are automatically approved.
* High-value transactions (> $500) are paused with a safety trigger requiring human manager authorization.

In [ ]:
# Test A: Small refund ($120) -> Cleared automatically
small_res = tools_mod.process_refund("ORD-2026-1004", refund_amount=120.0, reason="Late delivery perk")
print(f"🟢 Small Refund: Status = {small_res['status']} | Code = {small_res.get('confirmation_code')}")

# Test B: Large refund ($2,974.15) -> Requires Human Manager
large_res = tools_mod.process_refund("ORD-2026-1001", refund_amount=2974.15, reason="MacBook Return")
print(f"🟡 Large Refund: Status = {large_res['status']}")
print(f"   Trigger: {large_res.get('hitl_trigger')}")
print(f"   Action Required: {large_res.get('action_required')}")

## 3. Run the Autonomous ReAct Agent Loop

The ReAct loop executes autonomously: it thinks about what it needs, executes the right tools in order, and arrives at a complete conclusion with loop-breaker protection (`max_steps=5`).

In [ ]:
query = "Customer wants to return order ORD-2026-1001. Please inspect order eligibility and process refund."
agent_result = react_mod.run_react_agent(query, max_steps=5)

print("=" * 80)
print(f"🔎 User Request: {agent_result['query']}")
print("=" * 80)

for s in agent_result['steps']:
    print(f"\n🔹 Step {s['step']}:")
    print(f"   Thought    : {s['thought']}")
    print(f"   Action     : {s['action']}")
    print(f"   Observation: {s['observation'][:120]}...")

print("\n" + "=" * 80)
print(f"💡 Final Answer:\n{agent_result['final_answer']}")
print(f"⚠️ Human-in-the-Loop Triggered: {agent_result['hitl_triggered']}")
print("=" * 80)

## 4. Plan-and-Solve & Self-Reflection (Reflexion)

1. **Decomposes** the goal into sequential subtasks.
2. **Executes** each step in order.
3. **Reflects** on its own answers to catch policy violations (like runtime hour overages) before sending to the customer.

In [ ]:
goal = "Process return request for order ORD-2026-1001."

# 1. Generate & execute plan
plan_execution = plan_mod.execute_plan(goal)
print(f"📋 Generated Plan ({len(plan_execution['plan'])} steps):")
for step in plan_execution['plan']:
    print(f"   * {step}")

# 2. Self-Reflection & Critique
reflection = plan_mod.reflect_and_correct(goal, plan_execution)
print(f"\n🧠 Self-Reflection Valid: {reflection['is_valid']}")
print(f"   Recommendation: {reflection['final_recommendation']}")
print("   Critique Points:")
for point in reflection['critique_points']:
    print(f"     - {point}")